In [ ]:
from langgraph.graph import StateGraph , START , END 
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage , AIMessage 
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]

In [ ]:
from groq import Groq
def chat_node(state : ChatState) -> ChatState:

    messages = state['messages']

    groq_messages = []
    for msg in messages:
        if isinstance(msg , HumanMessage):
            groq_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            groq_messages.append({"role": "assistant", "content": msg.content})
    client = Groq(api_key="YOUR_GROQ_API_KEY")
    
    completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=groq_messages,
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)
    answer = ""
    for chunk in completion:
        answer += chunk.choices[0].delta.content or ""

    return {"messages": [AIMessage(content=answer)]}


In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node' , chat_node)

graph.add_edge(START , 'chat_node')
graph.add_edge('chat_node' , END)

In [ ]:
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# initial_state = {
#     'messages' : [HumanMessage(content ='What is the capital of india')]
# }

In [ ]:
# workflow.invoke(initial_state)

In [ ]:
thread_id = '1'
# every user will have his/her own thread id , its like user id in Databases , each user has their own for their convos

while True:

    user_msg = input("Type here : ")

    if user_msg.strip().lower() in ['exit' , 'quit' , 'bye']:
        break

    config = {'configurable' : {'thread_id' : thread_id}}


    response = workflow.invoke({'messages' : [HumanMessage(content = user_msg)]} , config=config)

    print('AI:' , response['messages'][-1].content)


AI: Hello, Swaraj! How can I assist you today?
AI: Your name is Swaraj.


In [ ]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hello my name is swaraj\\', additional_kwargs={}, response_metadata={}, id='da00f540-0be1-48ea-b9da-393de097816d'), AIMessage(content='Hello, Swaraj! How can I assist you today?', additional_kwargs={}, response_metadata={}, id='cce26650-4ee3-44de-a32d-a7a5ea4cc45b'), HumanMessage(content='what was my name?', additional_kwargs={}, response_metadata={}, id='0aceb184-e87a-49c1-bd28-f082ef625a55'), AIMessage(content='Your name is Swaraj.', additional_kwargs={}, response_metadata={}, id='64d423c4-9bc9-4de6-84d7-b3343b1b33f7')]}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aec08-db9c-6840-8004-a8ffef2fa84c'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2025-10-21T20:57:28.320197+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0aec08-d9ea-6022-8003-fa53006fe4b5'}}, tasks=(), interrupts=())

In [ ]:
# here the state is stored in the RAM , whenever workflow starts again , langgraph takes the past state from the RAM appends it and sends it to llm , it appends it because of the reducer